### Load libraries

In [ ]:
import apoc
import numpy as np
import tifffile as tiff
import matplotlib.pyplot as plt
import os 
from skimage import io
import time
import pandas as pd
from IPython.display import HTML

### Find files

First load the Classifier

In [ ]:
cl_filename = r"YOUR_PATH\CLASSIFIER_NAME"

Now choose the images you want to look at

In [ ]:
base_path = r"YOUR_PATH"

probe = "NR7_1nM_plus_benzo"

# find all cycle folders
cycle_folders = [f for f in os.listdir(base_path) if probe in f]
cycle_folders = sorted(cycle_folders)

print(cycle_folders)

### Generate Masks

The masks are saved as an additional layer inside the frame folders. To check how accurate the prediction is, we can display all masks and compare them to the original images eg. opened in an image viewer like Qupath.

In [ ]:
for cycle in cycle_folders:
    print(f"Start Cycle: {cycle}")
    cycle_path = os.path.join(base_path, cycle)
    frame_folders = [f for f in os.listdir(cycle_path) if f.endswith(".frames")]

    total_frames = len(frame_folders)

    for i, frame in enumerate(frame_folders, start=1):
        if not [frame_folders]:
            continue
        frame_path = os.path.join(cycle_path, frame)
        dapi, lyso, endo, nr = None, None, None, None
        print(f"[{cycle}] Frame {i}/{total_frames} ({i/total_frames:.0%}) → {frame}")

        
                # --- load channels ---
        for file in os.listdir(frame_path):
            full_path = os.path.join(frame_path, file)
            
            if "C001" in file:
                dapi = io.imread(full_path)
            elif "C002" in file:
                lyso = io.imread(full_path)
            elif "C003" in file:
                endo = tiff.imread(full_path)
            elif "C004" in file:
                nr = io.imread(full_path)
        
        if any(x is None for x in (dapi, lyso, endo, nr)):
            print("missing channel:", frame)
            continue

        sample_name= frame.replace(".tif.frames", "")
        
        result = classifier.predict(image=[dapi, lyso, endo, nr])
        prob_map = classifier.predict(image=[dapi, lyso, endo, nr])

        print("result", result.dtype, result.shape)
        plt.figure()
        plt.imshow(result, cmap= "gist_rainbow",  vmin=0, vmax=5)
        plt.show()
      
        
        tiff.imwrite(os.path.join(frame_path, f"{sample_name}_mask.tif"), result)

        # small GPU breathing room
        time.sleep(0.05)

print("done")